# B5: Qwen2.5-7B zero-shot baseline (separate experiment)

**Group 73, CITS4012.** A larger pretrained baseline, run by the group, kept in this separate notebook so that the main notebook (`CITS4012_73.ipynb`), the report and `submission/` are unchanged.

* **Baseline only.** The main model (DACT) is trained from scratch; this pretrained LLM is a baseline, as the brief allows (open-source model, our own experiment).
* **Same protocol as B4.** Same data split (`load_piqa`, seed 42), same prompt `Goal: {goal}\nSolution:` followed by each solution, same score (mean log-probability of the solution tokens, `score_continuations` from `src/baselines.py`), no training.
* **4-bit loading.** The 7B model does not fit a 16 GB T4 in fp16, so it is loaded with bitsandbytes NF4 (double quantisation, fp16 compute). This is part of the baseline's definition and is reported with it.
* **Test discipline.** Validation first. The test labels are read once, by `load_piqa_with_test_labels`, which appends the access to `test_access.log` in this experiment's folder; no decision about the main model uses this run.

Runtime: free Colab T4, about 30-40 min (model download included).

In [ ]:
import os, subprocess, sys
REPO_COMMIT = "d1e334a"   # code of main used for the split, the scoring and the evaluation helpers
if not os.path.isdir("/content/repo"):
    subprocess.run("git clone -q https://github.com/SEKY443/Group69TinyLanguageModel.git /content/repo", shell=True, check=True)
subprocess.run(f"git -C /content/repo checkout -q {REPO_COMMIT}", shell=True, check=True)
subprocess.run("pip -q install bitsandbytes gdown", shell=True, check=True)
sys.path.insert(0, "/content/repo/src")
os.environ["GROUP69_COMMIT"] = REPO_COMMIT
import importlib.metadata as md, torch
for p in ("torch", "transformers", "bitsandbytes", "accelerate"):
    try: print(f"{p:13s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:13s} not installed")
print("GPU          ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

torch         2.11.0+cu130
transformers  5.18.0
bitsandbytes  0.50.2
accelerate    1.15.0
GPU           Tesla T4


In [ ]:
import json, time
import numpy as np
from config import Config, JsonlLogger, environment_info
from data import load_piqa, load_piqa_with_test_labels, HIDDEN_LABEL
from baselines import score_continuations
from evaluate import accuracy, bootstrap_ci, mcnemar_exact, save_predictions

OUT = "/content/b5_qwen7b"
os.makedirs(OUT, exist_ok=True)
cfg = Config(data_dir="/content/repo/data/piqa")
train, val, test = load_piqa(cfg)
assert all(r["label"] == HIDDEN_LABEL for r in test)        # test labels are not in memory yet
print({"train": len(train), "val": len(val), "test": len(test)})

Downloaded ('train.jsonl', 'train-labels.lst', 'test.jsonl', 'test-labels.lst') from the unit's shared folder -> /content/repo/data/piqa
PIQA files: SHA-256 identical to the files used for the reported results
{'train': 14501, 'val': 1612, 'test': 1838}


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL = "Qwen/Qwen2.5-7B"
device = torch.device("cuda")
free, total = torch.cuda.mem_get_info(device)
print(f"GPU memory before loading: {free / 2**30:.1f} of {total / 2**30:.1f} GiB free")
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=quant, device_map={"": 0}).eval()
revision = getattr(model.config, "_commit_hash", None)
pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
log = JsonlLogger(f"{OUT}/logs/B5_zero_shot.jsonl")
log.log(event="start", model=MODEL, resolved_revision=revision, quantization="bitsandbytes NF4, double quant, fp16 compute",
        prompt="Goal: {goal}\nSolution: {solution}", score="mean log-probability of the solution tokens (<=128)",
        repo_commit=REPO_COMMIT, environment=environment_info(device))
print("loaded", MODEL, "revision", revision, f"| {torch.cuda.memory_allocated() / 2**30:.1f} GiB allocated")

def zero_shot(rows, batch_size=8):
    """Identical to llm_zero_shot in src/baselines.py (prompt, truncation, score), with the model loaded above."""
    seqs = []
    for r in rows:
        prompt = tok(f"Goal: {r['goal']}\nSolution:", add_special_tokens=True).input_ids
        for k in ("sol1", "sol2"):
            cont = tok(" " + r[k], add_special_tokens=False).input_ids[:128]
            seqs.append((prompt + cont, len(cont)))
    scores, oom = score_continuations(model, seqs, batch_size, pad_id, device,
                                      progress=lambda i: print(f"  {i}/{len(seqs)}", flush=True))
    if oom:
        log.log(event="oom_fallback", batch_sizes=oom)
    scores = scores.view(-1, 2)
    return scores.argmax(-1).numpy(), scores.softmax(-1)[:, 1].numpy()

GPU memory before loading: 14.5 of 14.6 GiB free


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

loaded Qwen/Qwen2.5-7B revision None | 5.2 GiB allocated


In [ ]:
# Validation (the split used for every decision)
t0 = time.time()
yva = np.array([r["label"] for r in val])
val_pred, val_prob = zero_shot(val)
val_acc = accuracy(val_pred, yva)
save_predictions(f"{OUT}/predictions/B5_qwen7b_zero_shot_val.jsonl", val, {"pred": val_pred, "prob": val_prob},
                 "B5_qwen7b_zero_shot", "val")
log.log(event="val", val_acc=val_acc, n=len(val), seconds=round(time.time() - t0, 1))
print(f"B5 Qwen2.5-7B zero-shot: validation accuracy {val_acc:.4f} ({time.time() - t0:.0f} s)")

  0/3224


  400/3224


  800/3224


  1200/3224


  1600/3224


  2000/3224


  2400/3224


  2800/3224


  3200/3224


B5 Qwen2.5-7B zero-shot: validation accuracy 0.8176 (356 s)


In [ ]:
# Final evaluation: the test labels are read once here, and the read is logged in OUT/test_access.log
import gzip
t0 = time.time()
_, _, test = load_piqa_with_test_labels(cfg, "B5 Qwen2.5-7B zero-shot baseline: final evaluation", OUT)
yte = np.array([r["label"] for r in test])
test_pred, test_prob = zero_shot(test)
test_acc = accuracy(test_pred, yte)
lo, hi = bootstrap_ci(test_pred, yte)
save_predictions(f"{OUT}/predictions/B5_qwen7b_zero_shot_test.jsonl", test, {"pred": test_pred, "prob": test_prob},
                 "B5_qwen7b_zero_shot", "test")
# McNemar against DACT's best-validation seed, as for every baseline in the report
dval = json.load(open("/content/repo/outputs/results/dact_full_val.json"))
best_seed = max(dval["runs"], key=lambda r: r["val_acc"])["seed"]
with gzip.open(f"/content/repo/outputs/predictions/dact_full_seed{best_seed}_test.jsonl.gz", "rt", encoding="utf-8") as f:
    recs = sorted((json.loads(l) for l in f if l.strip()), key=lambda r: r["index"])
assert [r["gold_label"] for r in recs] == yte.tolist()
dact_pred = np.array([r["predicted_label"] for r in recs])
b, c, p = mcnemar_exact(dact_pred, test_pred, yte)
result = {"model": MODEL, "revision": revision, "quantization": "NF4 4-bit", "val_acc": val_acc, "test_acc": test_acc,
          "test_ci95": [lo, hi], "mcnemar_vs_dact": {"dact_seed": best_seed, "only_dact": int(b), "only_b5": int(c), "p": p},
          "repo_commit": REPO_COMMIT}
json.dump(result, open(f"{OUT}/B5_results.json", "w"), indent=2)
log.log(event="test", **{k: v for k, v in result.items() if k not in ("model", "revision")})
print(json.dumps(result, indent=2))
print(open(f"{OUT}/test_access.log").read())

PIQA files: SHA-256 identical to the files used for the reported results


  0/3676


  400/3676


  800/3676


  1200/3676


...

  2400/3676


  2800/3676


  3200/3676


  3600/3676


{
  "model": "Qwen/Qwen2.5-7B",
  "revision": null,
  "quantization": "NF4 4-bit",
  "val_acc": 0.8176178660049628,
  "test_acc": 0.794341675734494,
  "test_ci95": [
    0.7758433079434167,
    0.8122959738846572
  ],
  "mcnemar_vs_dact": {
    "dact_seed": 42,
    "only_dact": 216,
    "only_b5": 540,
    "p": 7.530883698389237e-33
  },
  "repo_commit": "d1e334a"
}
{"time": "2026-10-07 14:00:02", "commit": "unknown", "reason": "B5 Qwen2.5-7B zero-shot baseline: final evaluation", "n_items": 1838}

